In [1]:
pip install torch transformers PyMuPDF nltk

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: C:\Users\Admin\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [3]:
"""
FinBERT-based ABSA pipeline for HBL ANNUAL reports
"""

import fitz
import csv
import re
from pathlib import Path

import nltk
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# --------------------------------------------------
# CONFIG
# --------------------------------------------------

INPUT_FOLDER = "AnnualReports"
OUTPUT_CSV = "Annual_report_finbert_absa.csv"
MODEL_NAME = "yiyanghkust/finbert-tone"

# --------------------------------------------------
# Aspect → Keyword Mapping
# --------------------------------------------------

aspects_keywords = {
    "profit": ["profit", "earnings", "income", "revenue"],
    "deposits": ["deposit", "casa", "savings"],
    "advances": ["loan", "lending", "advances"],
    "capital": ["capital", "capital adequacy", "cet1"],
    "npl": ["non-performing", "npl", "asset quality"],
    "provisions": ["provision", "impairment"],
    "risk": ["risk", "volatility", "stress"],
    "outlook": ["outlook", "expected", "forecast"],
    "macroeconomic": ["inflation", "interest rate", "policy rate"],
    "costs": ["cost", "expense", "cost-to-income"]
}

ALL_KEYWORDS = list({kw for v in aspects_keywords.values() for kw in v})

# --------------------------------------------------
# Exclusions (annual-report critical)
# --------------------------------------------------

EXCLUDED_SECTIONS = [
    "vision", "mission", "values",
    "sustainability", "corporate governance",
    "board of directors", "awards", "csr"
]

# --------------------------------------------------
# Utilities
# --------------------------------------------------

def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    return "\n".join(page.get_text() for page in doc if page.get_text())


def tokenize_sentences(text):
    try:
        nltk.data.find("tokenizers/punkt")
    except LookupError:
        nltk.download("punkt")
    return nltk.sent_tokenize(text)


def is_valid_sentence(s):
    return len(s.strip()) > 50 and not s.strip().startswith("•")


def is_excluded(s):
    s = s.lower()
    return any(bad in s for bad in EXCLUDED_SECTIONS)


# --------------------------------------------------
# FinBERT Setup
# --------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.eval()

LABEL_MAP = {0: -1, 1: 0, 2: 1}  # negative, neutral, positive


def finbert_score(sentence):
    inputs = tokenizer(
        sentence,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=1)[0]
    score = sum(LABEL_MAP[i] * probs[i].item() for i in range(3))
    return score


# --------------------------------------------------
# Aspect Scoring
# --------------------------------------------------

def score_aspects(sentences):
    scores = {a: [] for a in aspects_keywords}

    for s in sentences:
        s_low = s.lower()
        sent_score = finbert_score(s)

        for aspect, kws in aspects_keywords.items():
            if any(kw in s_low for kw in kws):
                scores[aspect].append(sent_score)

    return {
        a: (sum(vals) / len(vals) if vals else 0.0)
        for a, vals in scores.items()
    }


def calculate_wasi(aspect_scores):
    return sum(aspect_scores.values()) / len(aspect_scores)


# --------------------------------------------------
# Main
# --------------------------------------------------

def process_reports():
    input_path = Path(INPUT_FOLDER)
    results = []

    for pdf in input_path.glob("*.pdf"):
        print(f"Processing: {pdf.name}")

        text = extract_text_from_pdf(str(pdf))
        sentences = tokenize_sentences(text)

        filtered = [
            s for s in sentences
            if any(k in s.lower() for k in ALL_KEYWORDS)
            and not is_excluded(s)
            and is_valid_sentence(s)
        ]

        if not filtered:
            continue

        aspect_scores = score_aspects(filtered)
        wasi = calculate_wasi(aspect_scores)

        results.append([
            pdf.name,
            f"{wasi:.6f}",
            *[f"{aspect_scores[a]:.6f}" for a in aspects_keywords],
            len(filtered)
        ])

        print(f"  Sentences: {len(filtered)} | WASI: {wasi:.4f}")

    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "report_name",
            "overall_wasi",
            *[f"{a}_sentiment" for a in aspects_keywords],
            "num_sentences"
        ])
        writer.writerows(results)

    print(f"\nSaved to {OUTPUT_CSV}")


if __name__ == "__main__":
    process_reports()


Processing: HBL-Annual-Report-2015.pdf
  Sentences: 631 | WASI: -0.8115
Processing: HBL_Annual_Report_2016.pdf
  Sentences: 691 | WASI: -0.8080
Processing: HBL_Annual_Report_2017.pdf
  Sentences: 864 | WASI: -0.7591
Processing: HBL_Annual_Report_2018.pdf
  Sentences: 831 | WASI: -0.6990
Processing: HBL_Annual_Report_2019.pdf
  Sentences: 978 | WASI: -0.7741
Processing: HBL_Annual_Report_2020.pdf
  Sentences: 1020 | WASI: -0.7493
Processing: HBL_Annual_Report_2021.pdf
  Sentences: 1031 | WASI: -0.6823
Processing: HBL_Annual_Report_2022.pdf
  Sentences: 1055 | WASI: -0.7987
Processing: HBL_Annual_Report_2023.pdf
  Sentences: 1064 | WASI: -0.7158
Processing: HBL_Annual_Report_2024.pdf
  Sentences: 1082 | WASI: -0.7134

Saved to Annual_report_finbert_absa.csv
